# pbulk-cuda: Interim Report 2 experiments

**How to run:** Runtime > Change runtime type > **T4 GPU**, then Runtime > **Run all**. About fifteen minutes.

Runs the unit tests, then `bench/ir2.py`: the GPU layout build against the host one (idea 1), the five
kernels and cuSPARSE on three synthetic sizes, a block-size sweep, end to end with transfers, and Nsight
Compute on the medium size. Writes `ir2_results.json` and offers it as a download.


In [ ]:
import os, pathlib, shutil, subprocess, sys

# Leave empty to use the code stored in this notebook. Set to a git URL to clone the repository instead.
REPO_URL = ""

# Only for testing this notebook on a machine without a GPU. Leave unset on Colab.
EXTRA_ARGS = os.environ.get("PBULK_IR1_ARGS", "").split()
NO_GPU_OK = bool(EXTRA_ARGS)

if shutil.which("nvidia-smi") is None and not NO_GPU_OK:
    raise SystemExit("No GPU attached. Runtime > Change runtime type > T4 GPU, then Run all.")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                         capture_output=True, text=True).stdout)


def run(cmd, **kw):
    """Run a command and show its output as it comes."""
    print("$", " ".join(map(str, cmd)), flush=True)
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, **kw)
    lines = []
    for line in p.stdout:
        print(line, end="", flush=True)
        lines.append(line)
    return p.wait(), "".join(lines)


In [ ]:
# The code of the repository, packed by notebooks/build_notebook.py. Generated, do not edit.
SNAPSHOT = "..."


In [ ]:
import base64, io, tarfile

WORK = pathlib.Path(os.environ.get("PBULK_IR1_WORKDIR", "/content/pbulk-cuda-ir2"))
if REPO_URL:
    if not (WORK / ".git").exists():
        code, _ = run(["git", "clone", "--depth", "1", REPO_URL, WORK])
        assert code == 0, "git clone failed"
    print("using a clone of", REPO_URL)
else:
    WORK.mkdir(parents=True, exist_ok=True)
    with tarfile.open(fileobj=io.BytesIO(base64.b64decode(SNAPSHOT)), mode="r:gz") as tar:
        tar.extractall(WORK)
    print("unpacked the code snapshot stored in this notebook into", WORK)
os.chdir(WORK)
RESULTS = WORK / "results" / "ir2_results.json"
print(sorted(p.name for p in WORK.iterdir()))


In [ ]:
# Colab ships CuPy, NumPy, SciPy and pandas. Install only what is missing.
import importlib.util

need = [m for m in ("numpy", "scipy", "pandas", "pytest") if importlib.util.find_spec(m) is None]
if importlib.util.find_spec("cupy") is None and not NO_GPU_OK:
    need.append("cupy-cuda12x")
if need:
    run([sys.executable, "-m", "pip", "install", "-q", *need])
if not NO_GPU_OK:
    import cupy
    print("CuPy", cupy.__version__, "on", cupy.cuda.runtime.getDeviceProperties(0)["name"].decode())


In [ ]:
# Unit tests: CPU references, the C++ baseline, and both kernels in the CPU emulator.
test_code, test_out = run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider"])
test_summary = test_out.strip().splitlines()[-1] if test_out.strip() else ""
print("pytest exit code", test_code)


In [ ]:
harness_code, _ = run([sys.executable, "-m", "bench.ir2", "--out", RESULTS, *EXTRA_ARGS, '--sizes', 'large', '--stages', 'kernel,ncu', '--ncu-size', 'large', '--trials', '8'])
print("harness exit code", harness_code)


In [ ]:
import json
results = json.loads(RESULTS.read_text())
results["tests"] = {"pytest_exit": test_code, "summary": test_summary}
RESULTS.write_text(json.dumps(results, indent=1))
print("errors:", len(results["errors"]))
for e in results["errors"]:
    print(e["stage"]); print(e["trace"][-800:])
print("ncu available:", results.get("ncu", {}).get("available"))


In [ ]:
# The results file as gzip + base64 between markers, for copying out of the page if the download is blocked.
import base64, gzip
print("BEGIN_IR2_RESULTS")
print(base64.b64encode(gzip.compress(RESULTS.read_bytes())).decode())
print("END_IR2_RESULTS")


In [ ]:
print("results file:", RESULTS)
try:
    from google.colab import files
    files.download(str(RESULTS))
except Exception as e:      # not on Colab
    print("no browser download here:", type(e).__name__)
